# 01 · Basic RAG Pipeline
### Practical LLM Engineering — Module 04: RAG Systems

> **Objectives**
> - The complete RAG architecture: offline indexing + online retrieval + generation
> - Building a minimal but production-structured RAG pipeline from scratch
> - Prompt assembly: context formatting, citation injection, and length budgeting
> - Measuring end-to-end faithfulness and relevance
> - Engineering insights: latency decomposition and failure modes

---


## 1. Overview

**Retrieval-Augmented Generation (RAG)** grounds LLM responses in retrieved external knowledge, reducing hallucination and enabling up-to-date answers without retraining.

```
┌─────────── OFFLINE (once) ──────────────────────────────┐
│  Raw documents                                           │
│       ↓ chunk                                            │
│  Text chunks  ──embed──▶  Vectors  ──index──▶  VectorDB │
└──────────────────────────────────────────────────────────┘

┌─────────── ONLINE (per query) ──────────────────────────┐
│  User query ──embed──▶ VectorDB ──top-k──▶ Chunks       │
│                                                ↓         │
│  [System prompt + Context chunks + Question]  →  LLM    │
│                                                ↓         │
│                                           Answer         │
└──────────────────────────────────────────────────────────┘
```

### Why RAG beats pure LLM

| Problem with pure LLM | RAG solution |
|---|---|
| Knowledge cutoff | Retrieve from updated index |
| Hallucination | Ground answer in retrieved passages |
| No source attribution | Return chunk IDs as citations |
| Context too large to fine-tune | Dynamically inject relevant context |


## 2. Setup

In [ ]:
# !pip install numpy matplotlib scikit-learn -q
# ── Shared utilities (carried from Module 03) ────────────────────────
import os, re, json, time, math, random, textwrap
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict
from sklearn.preprocessing import normalize

# ── Mock embedder ─────────────────────────────────────────────────────
class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

# ── BM25 ──────────────────────────────────────────────────────────────
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self._corpus, self._doc_freqs, self._avgdl, self._N = [], defaultdict(int), 0.0, 0
    def _tok(self, t): return re.findall(r"\b\w+\b", t.lower())
    def fit(self, docs):
        self._corpus = [self._tok(d) for d in docs]; self._N = len(self._corpus)
        self._avgdl  = np.mean([len(d) for d in self._corpus])
        for doc in self._corpus:
            for t in set(doc): self._doc_freqs[t] += 1
    def _idf(self, t):
        n = self._doc_freqs.get(t, 0)
        return math.log((self._N - n + 0.5) / (n + 0.5) + 1)
    def get_scores(self, query):
        qt = self._tok(query)
        scores = []
        for doc in self._corpus:
            dl = len(doc); tf = defaultdict(int)
            for t in doc: tf[t] += 1
            s = sum(self._idf(t) * tf[t] * (self.k1+1) /
                    (tf[t] + self.k1*(1-self.b+self.b*dl/self._avgdl))
                    for t in qt if t in tf)
            scores.append(s)
        return np.array(scores)
    def get_top_k(self, query, k=5):
        s = self.get_scores(query); idx = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in idx]

# ── Vector store ──────────────────────────────────────────────────────
@dataclass
class Doc:
    id: str; text: str; metadata: dict = field(default_factory=dict)

class VectorStore:
    def __init__(self, embedder, dim=64):
        self.embedder = embedder
        self._vecs: np.ndarray = np.zeros((0, dim), dtype=np.float32)
        self._docs: list[Doc]  = []
    def add(self, docs: list[Doc]):
        vecs = self.embedder.embed([d.text for d in docs])
        self._vecs = np.vstack([self._vecs, vecs]) if len(self._vecs) else vecs
        self._docs.extend(docs)
    def search(self, query: str, k=5, where=None) -> list[tuple[Doc, float]]:
        if not self._docs: return []
        q = self.embedder.embed([query])[0]
        scores = self._vecs @ q
        if where:
            for i, doc in enumerate(self._docs):
                for key, val in where.items():
                    if doc.metadata.get(key) != val: scores[i] = -999
        top = np.argsort(-scores)[:k]
        return [(self._docs[i], float(scores[i])) for i in top]
    def __len__(self): return len(self._docs)

# ── Mock LLM ──────────────────────────────────────────────────────────
@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    """Returns templated answers grounded in provided context."""
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, system: str, user: str,
                 max_tokens=512, temperature=0.0) -> LLMResponse:
        time.sleep(0.02)
        ctx_match = re.search(r"Context:(.*?)(?:Question:|$)", user, re.DOTALL)
        ctx = ctx_match.group(1).strip()[:200] if ctx_match else ""
        q   = re.search(r"Question:(.*?)$", user, re.DOTALL)
        q   = q.group(1).strip()[:80] if q else user[:80]
        answer = (f"Based on the provided context, {q.lower().rstrip('?')} "
                  f"can be understood as follows: {ctx[:120]}... "
                  f"This answer is grounded in the retrieved documents.")
        n_in  = len((system+user).split())
        n_out = len(answer.split())
        return LLMResponse(answer, n_in, n_out, 45.0)
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

embedder = MockEmbedder(dim=64)
llm      = MockLLM()
print("✅ Shared utilities loaded (embedder + BM25 + VectorStore + MockLLM)")


## 3. The Knowledge Base

In [ ]:
KNOWLEDGE_BASE = [
    Doc("kb001", "Transformers use self-attention to compute weighted sums of value vectors, guided by query-key similarity scores scaled by √d_k.",
        {"topic": "ml", "subtopic": "transformers"}),
    Doc("kb002", "BERT is a bidirectional transformer pre-trained with masked language modelling and next-sentence prediction objectives.",
        {"topic": "ml", "subtopic": "bert"}),
    Doc("kb003", "RAG combines a retrieval system with a generative model, grounding outputs in retrieved documents to reduce hallucination.",
        {"topic": "rag", "subtopic": "overview"}),
    Doc("kb004", "Vector databases store dense embeddings and support approximate nearest-neighbour search using HNSW or IVF indexes.",
        {"topic": "retrieval", "subtopic": "vector_db"}),
    Doc("kb005", "BM25 is a sparse retrieval algorithm based on TF-IDF with length normalisation, widely used in search engines.",
        {"topic": "retrieval", "subtopic": "bm25"}),
    Doc("kb006", "Chunking divides documents into smaller pieces before embedding; chunk size balances embedding quality vs. retrieval precision.",
        {"topic": "rag", "subtopic": "chunking"}),
    Doc("kb007", "Hybrid search combines dense vector retrieval and sparse BM25 to improve recall across both semantic and keyword queries.",
        {"topic": "retrieval", "subtopic": "hybrid"}),
    Doc("kb008", "Fine-tuning adapts a pre-trained model to a specific task using supervised examples; LoRA reduces trainable parameters via low-rank decomposition.",
        {"topic": "ml", "subtopic": "finetuning"}),
    Doc("kb009", "Prompt engineering shapes LLM behaviour through system instructions, few-shot examples, and chain-of-thought reasoning.",
        {"topic": "prompting", "subtopic": "overview"}),
    Doc("kb010", "KV cache stores previously computed key-value pairs during autoregressive generation, avoiding redundant computation for past tokens.",
        {"topic": "inference", "subtopic": "kv_cache"}),
    Doc("kb011", "Reciprocal Rank Fusion (RRF) merges ranked retrieval lists without score normalisation, using 1/(k+rank) as a weight.",
        {"topic": "retrieval", "subtopic": "rrf"}),
    Doc("kb012", "Cross-encoders rerank retrieved candidates by scoring query-document pairs jointly, providing higher accuracy than bi-encoders.",
        {"topic": "retrieval", "subtopic": "reranking"}),
]

store = VectorStore(embedder, dim=embedder.dim)
store.add(KNOWLEDGE_BASE)
print(f"Knowledge base: {len(store)} documents indexed")


## 4. The RAG Pipeline

In [ ]:
RAG_SYSTEM_PROMPT = """You are a knowledgeable AI assistant. Answer questions using ONLY the provided context.
If the context does not contain enough information, say "I don't have enough context to answer this."
Always cite your sources by referencing the document IDs provided in the context."""

def format_context(retrieved: list[tuple[Doc, float]],
                   max_context_tokens: int = 400) -> str:
    """Format retrieved docs into a context block for the LLM prompt."""
    lines  = []
    tokens = 0
    for doc, score in retrieved:
        entry   = f"[{doc.id}] (relevance={score:.2f}): {doc.text}"
        n_toks  = len(entry.split())
        if tokens + n_toks > max_context_tokens:
            break
        lines.append(entry)
        tokens += n_toks
    return "\n".join(lines)


def build_rag_prompt(question: str, retrieved: list[tuple[Doc, float]]) -> tuple[str, str]:
    """Assemble the final (system, user) prompt pair."""
    context = format_context(retrieved)
    user    = (
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer (cite document IDs):"
    )
    return RAG_SYSTEM_PROMPT, user


class BasicRAGPipeline:
    """
    End-to-end Basic RAG pipeline.
    Offline: index documents into vector store.
    Online:  retrieve → format → generate.
    """
    def __init__(self, vector_store: VectorStore, llm: MockLLM,
                 k: int = 3, max_context_tokens: int = 400):
        self.store   = vector_store
        self.llm     = llm
        self.k       = k
        self.max_ctx = max_context_tokens

    def query(self, question: str, verbose: bool = False) -> dict:
        t_start = time.perf_counter()

        # Step 1: Retrieve
        t0        = time.perf_counter()
        retrieved = self.store.search(question, k=self.k)
        t_retrieve = (time.perf_counter() - t0) * 1000

        # Step 2: Assemble prompt
        system, user = build_rag_prompt(question, retrieved)
        ctx_tokens   = len(format_context(retrieved).split())

        # Step 3: Generate
        t0     = time.perf_counter()
        resp   = self.llm(system, user, max_tokens=300)
        t_gen  = (time.perf_counter() - t0) * 1000

        total_ms = (time.perf_counter() - t_start) * 1000

        result = {
            "question":      question,
            "answer":        resp.text,
            "sources":       [doc.id for doc, _ in retrieved],
            "source_scores": [score for _, score in retrieved],
            "ctx_tokens":    ctx_tokens,
            "input_tokens":  resp.input_tokens,
            "output_tokens": resp.output_tokens,
            "latency_retrieve_ms": t_retrieve,
            "latency_generate_ms": t_gen,
            "latency_total_ms":    total_ms,
        }

        if verbose:
            print(f"Q: {question}")
            print(f"Sources: {result['sources']} (scores: {[f'{s:.2f}' for s in result['source_scores']]})")
            print(f"A: {resp.text[:120]}...")
            print(f"Latency: retrieve={t_retrieve:.0f}ms  gen={t_gen:.0f}ms  total={total_ms:.0f}ms")
            print(f"Tokens: {resp.input_tokens} in / {resp.output_tokens} out")
        return result

    def batch_query(self, questions: list[str]) -> list[dict]:
        return [self.query(q) for q in questions]


rag = BasicRAGPipeline(store, llm, k=3)

TEST_QUESTIONS = [
    "How does self-attention work in transformer models?",
    "What is the difference between BM25 and vector search?",
    "How does RAG reduce hallucination in LLMs?",
    "What is a KV cache and why is it useful?",
]

print("Basic RAG Pipeline — queries\n")
results = []
for q in TEST_QUESTIONS:
    r = rag.query(q, verbose=True)
    results.append(r)
    print()


## 5. Latency Decomposition

In [ ]:
# ── Visualise latency breakdown ──────────────────────────────────────
retrieve_ms = [r["latency_retrieve_ms"] for r in results]
generate_ms = [r["latency_generate_ms"] for r in results]
questions_short = [q[:30] + "..." for q in TEST_QUESTIONS]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("RAG Pipeline Latency Decomposition", fontsize=11)

x = range(len(results))
axes[0].bar(x, retrieve_ms, label="Retrieve", color="#3498db")
axes[0].bar(x, generate_ms, bottom=retrieve_ms, label="Generate", color="#e67e22")
axes[0].set_xticks(x); axes[0].set_xticklabels(questions_short, rotation=20, ha="right", fontsize=8)
axes[0].set_ylabel("Latency (ms)"); axes[0].set_title("Stacked latency per query")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

labels  = ["Retrieve", "Generate"]
totals  = [sum(retrieve_ms), sum(generate_ms)]
axes[1].pie(totals, labels=labels, autopct="%1.0f%%",
            colors=["#3498db", "#e67e22"], startangle=90)
axes[1].set_title("Fraction of total latency")

plt.tight_layout(); plt.show()

print(f"Avg retrieve : {np.mean(retrieve_ms):.0f}ms  ({np.mean(retrieve_ms)/np.mean([r['latency_total_ms'] for r in results]):.0%})")
print(f"Avg generate : {np.mean(generate_ms):.0f}ms  ({np.mean(generate_ms)/np.mean([r['latency_total_ms'] for r in results]):.0%})")


## 6. Failure Modes

### The five main failure modes of basic RAG

| Mode | Symptom | Root cause |
|---|---|---|
| **Retrieval miss** | Answer is wrong / hedged | Query embedding didn't match relevant chunks |
| **Context overflow** | Answer truncated / incoherent | Too many chunks, exceeded LLM context window |
| **Low-quality chunks** | Answer vague | Chunks too small / bad boundaries |
| **LLM hallucination** | Answer contradicts context | LLM ignores context, uses parametric knowledge |
| **Citation mismatch** | Wrong sources cited | LLM associates claim with wrong chunk ID |


In [ ]:
# ── Diagnose retrieval quality ───────────────────────────────────────
def diagnose_retrieval(results: list[dict], threshold: float = 0.3) -> dict:
    """Flag results with potentially poor retrieval."""
    issues = []
    for r in results:
        max_score = max(r["source_scores"]) if r["source_scores"] else 0
        if max_score < threshold:
            issues.append({
                "question": r["question"][:50],
                "issue":    f"Low top-1 score ({max_score:.3f} < {threshold})",
                "severity": "HIGH" if max_score < 0.2 else "MEDIUM"
            })
    return {
        "total_queries": len(results),
        "flagged":       len(issues),
        "flag_rate":     len(issues) / max(len(results), 1),
        "issues":        issues,
    }

diag = diagnose_retrieval(results)
print(f"Retrieval diagnosis: {diag['flagged']}/{diag['total_queries']} queries flagged ({diag['flag_rate']:.0%})")
for issue in diag["issues"]:
    print(f"  [{issue['severity']}] {issue['question']} — {issue['issue']}")
if not diag["issues"]:
    print("  ✅ No retrieval issues detected")


## 7. Key Takeaways

| Concept | Summary |
|---|---|
| **Two phases** | Offline indexing (once) + online retrieval-generation (per query) |
| **Context window** | Budget context tokens carefully — leave room for answer |
| **Citation injection** | Include doc IDs in context block so LLM can cite sources |
| **Latency budget** | Retrieval typically < 10ms; generation dominates total latency |
| **Failure diagnosis** | Monitor top-1 retrieval score as a proxy for retrieval quality |

